# Notebook B: Combining /accounts, /groups, and /users

**Purpose:** Pull records from three related FOLIO endpoints and join them into a
single analysis-ready DataFrame. This is the kind of cross-record combination the
Stripes UI doesn't do natively.

**Assumed relationships** (confirm against your instance — I have not verified these
field names against current Sunflower schema, so treat them as a starting hypothesis
to check, not a fact):
- `/accounts` records reference a user via a `userId` field.
- `/users` records reference a patron group via a `patronGroup` field (a group's `id`).
- `/groups` records have an `id` and a human-readable `group` name.

If any of those field names are wrong for your instance, the fix is just changing the
column name in the merge step below — the overall approach stays the same.

**How to use this notebook:** Each section has a markdown cell explaining the step,
followed by a code cell. `# TODO (FOLIO-specific)` marks spots to confirm/adjust
against your actual schema.


## 1. Environment setup


In [13]:
# !pip install pandas requests

import pandas as pd
import requests

pd.set_option('display.max_columns', None)


## 2. Login
### Note: This script references patron data, so the login credentials must be able to access user accounts in FOLIO. 


In [14]:

%run folio_auth.ipynb


Login succeeded. Token retrieved.


## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. **Confirm the pagination param names and the response envelope key names
against your instance** — I'm using common FOLIO conventions here, but I have not
verified them against current docs.


In [15]:
def fetch_all_records(endpoint, records_key, limit=100, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records

## 4. Pull data from each endpoint

In [16]:

accounts_raw = fetch_all_records("/accounts", query='status.name="Open" and remaining > 0', records_key="accounts")
groups_raw   = fetch_all_records("/groups",   records_key="usergroups")  
users_raw    = fetch_all_records("/users",    records_key="users")

print(f"accounts: {len(accounts_raw)}")
print(f"groups:   {len(groups_raw)}")
print(f"users:    {len(users_raw)}")


accounts: 313
groups:   13
users:    1536


In [17]:
accounts_df = pd.DataFrame(accounts_raw)
groups_df   = pd.DataFrame(groups_raw)
users_df    = pd.DataFrame(users_raw)

users_df.head()


,username,id,active,patronGroup,departments,proxyFor,personal,createdDate,updatedDate,metadata,preferredEmailCommunication,externalSystemId,barcode,type,customFields,expirationDate,tags,enrollmentDate
0,fs00001115,5598c475-9931-49b0-945a-b78c1b2f8bc7,True,749e4843-fe70-403b-b428-e8d471432377,[],[],"{'lastName': 'fs00001115', 'firstName': 'fs000...",2022-04-07T18:57:37.164+00:00,2022-04-07T18:57:37.164+00:00,{'createdDate': '2022-04-07T18:57:37.150+00:00...,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mabrahamson,bfb5c924-cf59-4fba-abd5-9af531b7cea7,True,cf99d0df-0c3b-42f6-9037-9e2c28bb771d,[],[],"{'lastName': 'Abrahamson', 'firstName': 'Micha...",2026-06-23T17:54:32.553+00:00,2026-06-23T17:54:32.553+00:00,{'createdDate': '2024-04-05T15:23:21.244+00:00...,[],mabrahamson@ebsco.com,1234123412341234,patron,"{'activePatronGroups': [], 'genero': 'opt_0', ...",NaN,NaN,NaN
2,ccrawford,4aeff477-2268-4652-af88-f922fb4df22d,True,b89ee7e1-8ca6-43bf-8ffa-6e6c3996b51d,[],[],"{'lastName': 'Crawford', 'firstName': 'Chloe',...",2026-02-23T22:17:31.817+00:00,2026-02-23T22:17:31.817+00:00,{'createdDate': '2022-04-12T22:35:55.612+00:00...,[],ccrawford,22874000505615,object,{'campusAffiliation': 'opt_0'},2027-02-01T23:59:59.000+00:00,{'tagList': []},NaN
3,fs00000002-system-user,0ef20940-e2da-4165-a4ea-608cdee8d99d,True,NaN,[],[],"{'lastName': 'System', 'firstName': 'System Us...",2025-10-27T11:17:23.464+00:00,2025-10-27T11:17:23.464+00:00,{'createdDate': '2025-10-27T11:17:23.445+00:00...,[],NaN,NaN,system,{},NaN,NaN,NaN
4,stagingDirector,617b1b93-0f4a-47bf-9b46-3c50071bd07c,True,749e4843-fe70-403b-b428-e8d471432377,[],[],"{'lastName': 'stagingDirector', 'firstName': '...",2022-04-07T19:00:51.561+00:00,2022-04-07T19:00:51.561+00:00,{'createdDate': '2022-04-07T19:00:51.547+00:00...,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?

In [18]:
print(accounts_df.columns.tolist())
print(users_df.columns.tolist())
print(groups_df.columns.tolist())


['amount', 'remaining', 'status', 'paymentStatus', 'feeFineType', 'feeFineOwner', 'callNumber', 'metadata', 'userId', 'feeFineId', 'ownerId', 'id', 'contributors', 'title', 'barcode', 'location', 'dueDate', 'loanId', 'itemId', 'materialTypeId', 'materialType', 'loanPolicyId', 'overdueFinePolicyId', 'lostItemFeePolicyId', 'holdingsRecordId', 'instanceId', 'returnedDate']
['username', 'id', 'active', 'patronGroup', 'departments', 'proxyFor', 'personal', 'createdDate', 'updatedDate', 'metadata', 'preferredEmailCommunication', 'externalSystemId', 'barcode', 'type', 'customFields', 'expirationDate', 'tags', 'enrollmentDate']
['group', 'desc', 'id', 'metadata', 'expirationOffsetInDays']


In [19]:
# Spot-check types of the columns you intend to join on
print(accounts_df['userId'].dtype)
print(users_df['id'].dtype)
print(users_df['patronGroup'].dtype)
print(groups_df['id'].dtype)


str
str
str
str


## 6. Merge

Two joins: accounts → users, then that result → groups.

Starting with `how='left'` keeps every account row even if a match isn't found, so you
can see what didn't match rather than silently losing rows.


In [20]:
# Step 1: accounts + users
accounts_users = accounts_df.merge(
    users_df,
    left_on='userId',
    right_on='id',
    how='left',
    suffixes=('_account', '_user'),
)

# Step 2: + groups
full_df = accounts_users.merge(
    groups_df,
    left_on='patronGroup',
    right_on='id',
    how='left',
    suffixes=('', '_group'),
)

full_df.head()


,amount,remaining,status,paymentStatus,feeFineType,feeFineOwner,callNumber,metadata_account,userId,feeFineId,ownerId,id_account,contributors,title,barcode_account,location,dueDate,loanId,itemId,materialTypeId,materialType,loanPolicyId,overdueFinePolicyId,lostItemFeePolicyId,holdingsRecordId,instanceId,returnedDate,username,id_user,active,patronGroup,departments,proxyFor,personal,createdDate,updatedDate,metadata_user,preferredEmailCommunication,externalSystemId,barcode_user,type,customFields,expirationDate,tags,enrollmentDate,group,desc,id,metadata,expirationOffsetInDays
0,2.0,2.0,{'name': 'Open'},{'name': 'Outstanding'},Replacement card,Birmingham Campus,,{'createdDate': '2023-02-08T21:43:21.434+00:00...,5de06195-de05-4190-ab4e-f2e69f66503e,993593ed-db1f-4cff-9271-69a5bcc7aab2,b4f7129d-6658-4e74-bf26-07d7dc1c551e,af22a561-46aa-4101-88a2-48eae6f7ff59,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bsmith,5de06195-de05-4190-ab4e-f2e69f66503e,True,3ddd748f-5ba1-46a6-87b2-1a45a7243624,[93697efb-4e44-4683-aa68-3c5e06d97195],[],"{'lastName': 'Smith', 'firstName': 'Betty', 'p...",2023-06-01T20:07:12.818+00:00,2024-02-07T17:46:48.765+00:00,{'createdDate': '2022-11-22T16:46:43.492+00:00...,[],bsmith@maildrop.cc,22874000520520,NaN,"{'studentId': '12345', 'campusAffiliation': 'o...",NaN,NaN,NaN,Faculty,Full time campus faculty,3ddd748f-5ba1-46a6-87b2-1a45a7243624,{'createdDate': '2022-04-11T15:58:52.965+00:00...,NaN
1,50.0,50.0,{'name': 'Open'},{'name': 'Refunded fully'},Lost item fee,Contoocook Campus,Needs call number,{'createdDate': '2022-10-13T04:31:25.585+00:00...,10c125df-9eaa-48c3-9295-a2cd17a37e54,cf238f9f-7018-47b7-b815-bb2db798e19f,476a7fb1-f289-449a-b300-15f8221b3d5e,f23f4269-cf42-4292-b28f-785beb6379e5,"[{'name': 'Santmyer, Helen Hooven, 1895-1986'}]","""---and ladies of the club"" / Helen Hooven San...",32260005504685,GOBI Stacks,2022-08-13T03:59:00.000+00:00,df4b454b-aa45-4562-b7ae-b2edaf9a0a29,d3dda022-1f05-46b4-a694-5a0af7b69a29,13493ed9-900c-4db2-8646-4798bd8ac213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hcunningham,10c125df-9eaa-48c3-9295-a2cd17a37e54,True,b89ee7e1-8ca6-43bf-8ffa-6e6c3996b51d,[],[],"{'lastName': 'Cunningham', 'firstName': 'Honey...",2026-02-23T22:17:31.549+00:00,2026-02-23T22:17:31.549+00:00,{'createdDate': '2022-04-12T22:35:52.081+00:00...,[],hcunningham,22874000505591,object,{'campusAffiliation': 'opt_0'},2027-02-01T23:59:59.000+00:00,{'tagList': []},NaN,Community,Local resident with library privileges,b89ee7e1-8ca6-43bf-8ffa-6e6c3996b51d,{'createdDate': '2022-04-11T15:58:52.963+00:00...,365.0
2,50.0,50.0,{'name': 'Open'},{'name': 'Outstanding'},Lost item fee,Birmingham Campus,PS3611.I44 I58 2014,{'createdDate': '2023-02-10T05:18:49.936+00:00...,5de06195-de05-4190-ab4e-f2e69f66503e,cf238f9f-7018-47b7-b815-bb2db798e19f,b4f7129d-6658-4e74-bf26-07d7dc1c551e,8089d9a2-9765-4df3-8481-a73ff5b0386e,"[{'name': 'Kidd, Sue Monk'}]",The invention of wings / Sue Monk Kidd.,32260010269159,EBSCONET 2nd Floor,2023-01-26T04:59:59.000+00:00,ddbcdddc-12ee-4c1d-b14c-8bb515275808,8ed9d33c-9519-4d91-b2e0-1e0a2076fbf3,13493ed9-900c-4db2-8646-4798bd8ac213,Book,c3033373-8b64-4931-8b4a-43902930b7b7,612518cf-8597-4685-8d53-b9a1e2053a94,cb18f7aa-8af4-4b6d-965c-ae61fe91626c,NaN,NaN,NaN,bsmith,5de06195-de05-4190-ab4e-f2e69f66503e,True,3ddd748f-5ba1-46a6-87b2-1a45a7243624,[93697efb-4e44-4683-aa68-3c5e06d97195],[],"{'lastName': 'Smith', 'firstName': 'Betty', 'p...",2023-06-01T20:07:12.818+00:00,2024-02-07T17:46:48.765+00:00,{'createdDate': '2022-11-22T16:46:43.492+00:00...,[],bsmith@maildrop.cc,22874000520520,NaN,"{'studentId': '12345', 'campusAffiliation': 'o...",NaN,NaN,NaN,Faculty,Full time campus faculty,3ddd748f-5ba1-46a6-87b2-1a45a7243624,{'createdDate': '2022-04-11T15:58:52.965+00:00...,NaN
3,10.0,10.0,{'name': 'Open'},{'name': 'Outstanding'},Lost item processing fee,Birmingham Campus,PS3611.I44 I58 2014,{'createdDate': '2023-02-10T05:18:49.935+00:00...,5de06195-de05-4190-ab4e-f2e69f66503e,c7dede15-aa48-45ed-860b-f996540180e0,b

## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [21]:
print("Original accounts rows:", len(accounts_df))
print("After merging with users:  ", len(accounts_users))
print("After merging with groups: ", len(full_df))

# Rows where the user or group match failed — worth investigating, not ignoring
unmatched_users = full_df[full_df['id_user'].isnull()] if 'id_user' in full_df.columns else pd.DataFrame()
print("Accounts with no matching user:", len(unmatched_users))


Original accounts rows: 313
After merging with users:   313
After merging with groups:  313
Accounts with no matching user: 0


## 8. Analyze the combined dataset

Now that accounts, users, and groups are joined, you can ask questions that span all
three — e.g. total fee/fine amounts by patron group. Adjust field names to match your
actual `/accounts` schema (commonly something like `amount` or `remaining`).


### This call creates a summary of total fines by patron group

In [22]:
summary = full_df.groupby('group')['remaining'].sum().sort_values(ascending=False)
print(summary)


group
Student                 3449.99
Staff                   2647.00
Faculty                 2564.00
Community                730.00
zEBSCO Support Group     674.00
MC Student               180.00
Book club                 12.00
Interlibrary loan         10.00
Reading Room Users         2.00
Name: remaining, dtype: float64


In [23]:
# fees_fines_report = full_df[['username', 'remaining', 'feeFineType']]
full_df.head()

,amount,remaining,status,paymentStatus,feeFineType,feeFineOwner,callNumber,metadata_account,userId,feeFineId,ownerId,id_account,contributors,title,barcode_account,location,dueDate,loanId,itemId,materialTypeId,materialType,loanPolicyId,overdueFinePolicyId,lostItemFeePolicyId,holdingsRecordId,instanceId,returnedDate,username,id_user,active,patronGroup,departments,proxyFor,personal,createdDate,updatedDate,metadata_user,preferredEmailCommunication,externalSystemId,barcode_user,type,customFields,expirationDate,tags,enrollmentDate,group,desc,id,metadata,expirationOffsetInDays
0,2.0,2.0,{'name': 'Open'},{'name': 'Outstanding'},Replacement card,Birmingham Campus,,{'createdDate': '2023-02-08T21:43:21.434+00:00...,5de06195-de05-4190-ab4e-f2e69f66503e,993593ed-db1f-4cff-9271-69a5bcc7aab2,b4f7129d-6658-4e74-bf26-07d7dc1c551e,af22a561-46aa-4101-88a2-48eae6f7ff59,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bsmith,5de06195-de05-4190-ab4e-f2e69f66503e,True,3ddd748f-5ba1-46a6-87b2-1a45a7243624,[93697efb-4e44-4683-aa68-3c5e06d97195],[],"{'lastName': 'Smith', 'firstName': 'Betty', 'p...",2023-06-01T20:07:12.818+00:00,2024-02-07T17:46:48.765+00:00,{'createdDate': '2022-11-22T16:46:43.492+00:00...,[],bsmith@maildrop.cc,22874000520520,NaN,"{'studentId': '12345', 'campusAffiliation': 'o...",NaN,NaN,NaN,Faculty,Full time campus faculty,3ddd748f-5ba1-46a6-87b2-1a45a7243624,{'createdDate': '2022-04-11T15:58:52.965+00:00...,NaN
1,50.0,50.0,{'name': 'Open'},{'name': 'Refunded fully'},Lost item fee,Contoocook Campus,Needs call number,{'createdDate': '2022-10-13T04:31:25.585+00:00...,10c125df-9eaa-48c3-9295-a2cd17a37e54,cf238f9f-7018-47b7-b815-bb2db798e19f,476a7fb1-f289-449a-b300-15f8221b3d5e,f23f4269-cf42-4292-b28f-785beb6379e5,"[{'name': 'Santmyer, Helen Hooven, 1895-1986'}]","""---and ladies of the club"" / Helen Hooven San...",32260005504685,GOBI Stacks,2022-08-13T03:59:00.000+00:00,df4b454b-aa45-4562-b7ae-b2edaf9a0a29,d3dda022-1f05-46b4-a694-5a0af7b69a29,13493ed9-900c-4db2-8646-4798bd8ac213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hcunningham,10c125df-9eaa-48c3-9295-a2cd17a37e54,True,b89ee7e1-8ca6-43bf-8ffa-6e6c3996b51d,[],[],"{'lastName': 'Cunningham', 'firstName': 'Honey...",2026-02-23T22:17:31.549+00:00,2026-02-23T22:17:31.549+00:00,{'createdDate': '2022-04-12T22:35:52.081+00:00...,[],hcunningham,22874000505591,object,{'campusAffiliation': 'opt_0'},2027-02-01T23:59:59.000+00:00,{'tagList': []},NaN,Community,Local resident with library privileges,b89ee7e1-8ca6-43bf-8ffa-6e6c3996b51d,{'createdDate': '2022-04-11T15:58:52.963+00:00...,365.0
2,50.0,50.0,{'name': 'Open'},{'name': 'Outstanding'},Lost item fee,Birmingham Campus,PS3611.I44 I58 2014,{'createdDate': '2023-02-10T05:18:49.936+00:00...,5de06195-de05-4190-ab4e-f2e69f66503e,cf238f9f-7018-47b7-b815-bb2db798e19f,b4f7129d-6658-4e74-bf26-07d7dc1c551e,8089d9a2-9765-4df3-8481-a73ff5b0386e,"[{'name': 'Kidd, Sue Monk'}]",The invention of wings / Sue Monk Kidd.,32260010269159,EBSCONET 2nd Floor,2023-01-26T04:59:59.000+00:00,ddbcdddc-12ee-4c1d-b14c-8bb515275808,8ed9d33c-9519-4d91-b2e0-1e0a2076fbf3,13493ed9-900c-4db2-8646-4798bd8ac213,Book,c3033373-8b64-4931-8b4a-43902930b7b7,612518cf-8597-4685-8d53-b9a1e2053a94,cb18f7aa-8af4-4b6d-965c-ae61fe91626c,NaN,NaN,NaN,bsmith,5de06195-de05-4190-ab4e-f2e69f66503e,True,3ddd748f-5ba1-46a6-87b2-1a45a7243624,[93697efb-4e44-4683-aa68-3c5e06d97195],[],"{'lastName': 'Smith', 'firstName': 'Betty', 'p...",2023-06-01T20:07:12.818+00:00,2024-02-07T17:46:48.765+00:00,{'createdDate': '2022-11-22T16:46:43.492+00:00...,[],bsmith@maildrop.cc,22874000520520,NaN,"{'studentId': '12345', 'campusAffiliation': 'o...",NaN,NaN,NaN,Faculty,Full time campus faculty,3ddd748f-5ba1-46a6-87b2-1a45a7243624,{'createdDate': '2022-04-11T15:58:52.965+00:00...,NaN
3,10.0,10.0,{'name': 'Open'},{'name': 'Outstanding'},Lost item processing fee,Birmingham Campus,PS3611.I44 I58 2014,{'createdDate': '2023-02-10T05:18:49.935+00:00...,5de06195-de05-4190-ab4e-f2e69f66503e,c7dede15-aa48-45ed-860b-f996540180e0,b

In [ ]:

fees_fines_report = full_df.loc[:, ['barcode_user','personal','username','amount','remaining', 'feeFineType', 'status','title']]
# user=full_df.personal[3]
# fees_fines_report.head()
fees_fines_report.head()

AttributeError: 'dict' object has no attribute 'head'